In [2]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study
import diff_cont
import lambda_cont
import libs.agent_infra as ai
import os

import json
import datetime
import itertools
import constants as Cs
import concurrent.futures
import numpy as np
from concurrent.futures import ProcessPoolExecutor

SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 6

/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-30 19:46:49.080934: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-30 19:46:49.175966: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-30 19:46:51.484852: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF

In [ ]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 0.7, step=0.1)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="add_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
study = create_study(direction="maximize", sampler=sampler, study_name="diff_add_novelty", storage="sqlite:///Data/optuna/lunarlander/add_novelty/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=170, n_jobs=5)
print(study.best_trials)

[I 2026-05-30 19:50:54,798] A new study created in RDB with name: diff_add_novelty


   	                            fitness                            	                            novelty                             
   	---------------------------------------------------------------	----------------------------------------------------------------
gen	avg     	gen	max      	min     	std    	avg     	gen	max     	min      	std     
0  	-316.864	0  	-0.171698	-608.907	144.144	0.369533	0  	0.731536	0.0133541	0.171938
   	                        fitness                        	                            novelty                             
   	-------------------------------------------------------	----------------------------------------------------------------
gen	avg     	gen	max     	min    	std    	avg     	gen	max     	min      	std     
0  	-355.947	0  	-57.0884	-616.56	156.157	0.372832	0  	0.720176	0.0117513	0.179193
   	                            fitness                            	                        novelty                         
   	-------------------

[I 2026-05-30 20:22:53,478] Trial 2 finished with value: -43.629528973095994 and parameters: {'lambda': 40, 'mutation_rate': 0.49, 'cross_rate': 1.0, 'start_fit_w': 0.30000000000000004}. Best is trial 2 with value: -43.629528973095994.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A 

13 	-354.412	13 	-54.9417	-730.36 	164.37 	0.389589	13 	0.632064	0.0174276 	0.140692
11 	-412.148	11 	-85.8552	-822.029	157.797	0.358709	11 	0.798945	0.0139452  	0.159951
14 	-403.157	14 	-6.40294	-712.513	179.119	0.318299	14 	0.636071	0.00102762 	0.162798
12 	-389.391	12 	-68.769 	-650.583	163.859	0.202596	12 	0.855231	0.0141662 	0.153104
14 	-386.477	14 	95.0708 	-712.897	189.608	0.221593	14 	0.840639	0.00241801 	0.184557
12 	-423.819	12 	-58.2703	-687.313	169.732	0.243521	12 	0.792504	0.0357552  	0.215755
13 	-351.6  	13 	-19.9199	-648.504	156.326	0.292684	13 	0.709064	0.0428563 	0.140262
12 	-369.477	12 	-84.5277	-747.94 	194.537	0.339224	12 	0.722524	0.0121172 	0.160581
13 	-364.854	13 	-99.9449	-664.793	154.691	0.259541	13 	0.829475	0.0117024  	0.16897 
14 	-382.555	14 	-6.86554	-708.458	178.525	0.259931	14 	0.837643	0.0139638 	0.192027
14 	-369.048	14 	19.4799 	-656.053	157.412	0.31354 	14 	0.68247 	0.00260547	0.160307
12 	-410.602	12 	-22.8992	-605.158	160.725	0.277648	12 	0.71

[I 2026-05-30 20:49:59,546] Trial 3 finished with value: -7.160407806308261 and parameters: {'lambda': 60, 'mutation_rate': 0.33, 'cross_rate': 1.0, 'start_fit_w': 0.7}. Best is trial 3 with value: -7.160407806308261.
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessNovelty' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/home/schkliba/.pyenv/versions/prometheus/lib/python3.10/site-packages/deap/creator.py:185: RuntimeWarning: A class named 'Fitne

...

11 	-383.477	11 	-125.116	-625.948	132.852	0.269118	11 	0.807928	0.0345993  	0.168367
13 	-416.359	13 	-85.1814	-724.11 	151.168	0.263758	13 	0.845626	0.0018121 	0.158737
14 	-328.946	14 	33.1999 	-597.342	154.107	0.283241	14 	0.810537	0.0710889	0.193686
12 	-393.141	12 	-114.771	-656.094	143.82 	0.312545	12 	0.86355 	0.0621913  	0.187015
14 	-404.899	14 	-6.89756	-712.807	191.767	0.249998	14 	0.738692	0.00573292	0.186952
13 	-397.959	13 	-29.6271	-689.415	172.627	0.265939	13 	0.763239	0.035688   	0.133951
14 	-389.043	14 	63.5993 	-587.289	164.525	0.250329	14 	0.740382	0.0230837  	0.185815


[I 2026-05-31 16:21:52,498] Trial 169 finished with value: 73.61062691414709 and parameters: {'lambda': 40, 'mutation_rate': 0.21, 'cross_rate': 0.4, 'start_fit_w': 0.6000000000000001}. Best is trial 69 with value: 87.73399179934741.
Process ForkProcess-178:
Process ForkProcess-179:
Process ForkProcess-358:
Process ForkProcess-446:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):


KeyboardInterrupt: 

  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)


  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/schkliba/.pyenv/versions/3.10.12/lib/python3.10/concurrent/futures/process.py", line 240, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/schkliba/.pyen